# CRAFT-GC: Stage B GPU Experiments (Google Colab)

**Setup (once):**
1. Runtime → **Change runtime type** → **T4 GPU**
2. **HF token** (pick one):
   - **Colab Secrets:** left sidebar → 🔑 **Secrets** → **+ Add secret** → Name exactly `HF_TOKEN` → paste token → toggle **Notebook access** ON
   - **Or** Cell 1 will prompt you to type the token (hidden input)
3. Run all cells. Stage B takes **2–4 hours**.

**After finish:** download `results/` and `craft-gc-paper/figures/`.

In [ ]:
!pip install -q torch torchvision open-clip-torch diffusers transformers accelerate pandas matplotlib seaborn huggingface_hub

import os, sys, subprocess
from getpass import getpass

def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception as e:
        print("Colab Secret HF_TOKEN not found — type token below (input hidden).")
        print("To use Secrets: sidebar 🔑 → Add secret → name HF_TOKEN → enable Notebook access.")
        return getpass("Hugging Face token: ")

from huggingface_hub import login
login(token=get_hf_token())

REPO_URL = "https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git"
PROJECT = "/content/Research"

if os.path.isdir(PROJECT):
    subprocess.run(["git", "-C", PROJECT, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT], check=True)

import torch
print("CWD:", os.getcwd())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
import craft_gc
print("craft_gc OK:", craft_gc.__version__)

In [ ]:
!python scripts/evaluate_embeddings.py --device cuda
!python scripts/merge_paper_results.py
!python scripts/plot_figures.py

In [ ]:
# Stage B — expect 2–4 HOURS. Must run Cell 1 first (clone + pip install -e).
import os, sys, subprocess

PROJECT = "/content/Research"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime → T4 GPU → Save → restart → re-run Cell 1")

# Ensure latest code from GitHub
subprocess.run(["git", "-C", PROJECT, "pull"], check=False)

script = f"{PROJECT}/scripts/colab_run_stage_b.py"
if not os.path.isfile(script):
    raise FileNotFoundError(
        f"{script} missing. Re-run Cell 1 or:\n"
        "!rm -rf /content/Research && !git clone https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git /content/Research"
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Running:", script)
print("This takes 2–4 hours (1000 images)...")

result = subprocess.run(
    [sys.executable, script, "--device", "cuda", "--limit", "50"],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Stage B failed (exit {result.returncode}). See output above.")
print("Done. Download results/ and craft-gc-paper/figures/")